# Does my IV strategy recover the true coefficient under weak instruments?

This notebook uses `spuriosity` to generate data with an **endogenous** regressor (correlated with the outcome's error term, biasing naive OLS) and a valid instrument. We compare naive OLS against two-stage least squares (2SLS) under both a **strong** and a **weak** instrument, to see what "use an instrument" actually buys you — and where it stops working.

## Setup

In [2]:
%pip install -q "spuriosity[linearmodels] @ git+https://github.com/Nityahapani/spuriosity.git"

Note: you may need to restart the kernel to use updated packages.


In [3]:
from spuriosity import PanelGenerator, reference
import numpy as np

print("spuriosity imported successfully")

spuriosity imported successfully


## Part 1: Strong instrument

We generate `education` as an endogenous regressor: a latent variable affects both `education` and the outcome `y` directly, and `distance_to_college` is a valid instrument (affects `education` but not `y` except through it).

In [5]:
gen_strong = PanelGenerator(n_entities=200000, n_periods=1, seed=42)
gen_strong.add_variable("education", dist="normal", mean=0, std=1)
gen_strong.set_outcome(formula="education", coefficients={"education": 2.0, "Intercept": 0.0}, noise_std=0.1)
gen_strong.add_endogeneity(
    feature="education", instrument="distance_to_college",
    instrument_strength=1.0, endogeneity_strength=1.0,
)
df_strong, truth_strong = gen_strong.generate()
df_strong.head()

,entity_id,period,education,distance_to_college,y
0,0,0,2.141552,1.648040,5.022445
1,1,0,0.992755,-0.324929,2.909573
2,2,0,-0.768780,-0.044616,-1.626796
3,3,0,-2.984696,-0.822605,-7.375435
4,4,0,-2.755549,-2.157146,-5.831767


In [6]:
print(truth_strong)

GroundTruth(seed=42, spuriosity='0.1.0')
  true_coefficients: {'education': 2.0, 'Intercept': 0.0}
  endogeneity: [('education', 'distance_to_college')]
  has_true_cate: False


In [7]:
fit_naive_strong = reference.ols_fit(df_strong, formula="y ~ education")
fit_iv_strong = reference.iv2sls_fit(df_strong, outcome="y", endogenous=["education"], instruments=["distance_to_college"])
f_stat_strong = reference.first_stage_f_stat(df_strong, endogenous="education", instruments=["distance_to_college"])

print("First-stage F-stat:", round(f_stat_strong, 1))
print("Naive OLS estimate:", round(fit_naive_strong.coefficients["education"], 3), " (true: 2.0)")
print("2SLS estimate:     ", round(fit_iv_strong.coefficients["education"], 3), " (true: 2.0)")

First-stage F-stat: 160998.5
Naive OLS estimate: 2.444  (true: 2.0)
2SLS estimate:      2.001  (true: 2.0)


Naive OLS is meaningfully biased (education is endogenous), but 2SLS using the instrument recovers the true coefficient almost exactly. The first-stage F-statistic is enormous — a very strong instrument.

## Part 2: Weak instrument

Same setup, but now `instrument_strength` is much smaller — the instrument barely moves `education`. This is the "weak instrument" scenario every applied IV paper is supposed to check for.

In [10]:
gen_weak = PanelGenerator(n_entities=200000, n_periods=1, seed=42)
gen_weak.add_variable("education", dist="normal", mean=0, std=1)
gen_weak.set_outcome(formula="education", coefficients={"education": 2.0, "Intercept": 0.0}, noise_std=0.1)
gen_weak.add_endogeneity(
    feature="education", instrument="distance_to_college",
    instrument_strength=0.005, endogeneity_strength=1.0,
)
df_weak, truth_weak = gen_weak.generate()

fit_naive_weak = reference.ols_fit(df_weak, formula="y ~ education")
fit_iv_weak = reference.iv2sls_fit(df_weak, outcome="y", endogenous=["education"], instruments=["distance_to_college"])
f_stat_weak = reference.first_stage_f_stat(df_weak, endogenous="education", instruments=["distance_to_college"])

print("First-stage F-stat:", round(f_stat_weak, 2), " (below 10 = weak, per the Stock-Yogo rule of thumb)")
print("Naive OLS estimate:", round(fit_naive_weak.coefficients["education"], 3), " (true: 2.0)")
print("2SLS estimate:     ", round(fit_iv_weak.coefficients["education"], 3), " (true: 2.0)")
print("2SLS standard error:", round(fit_iv_weak.raw_model.std_errors["education"], 3))

First-stage F-stat: 4.41  (below 10 = weak, per the Stock-Yogo rule of thumb)
Naive OLS estimate: 2.8  (true: 2.0)
2SLS estimate:      2.229  (true: 2.0)
2SLS standard error: 0.335


## Side-by-side comparison

In [12]:
print(f"{'':25} {'F-stat':>10} {'OLS est':>10} {'2SLS est':>10} {'2SLS SE':>10}")
print(f"{'Strong instrument':25} {f_stat_strong:>10.1f} {fit_naive_strong.coefficients['education']:>10.3f} {fit_iv_strong.coefficients['education']:>10.3f} {fit_iv_strong.raw_model.std_errors['education']:>10.4f}")
print(f"{'Weak instrument':25} {f_stat_weak:>10.2f} {fit_naive_weak.coefficients['education']:>10.3f} {fit_iv_weak.coefficients['education']:>10.3f} {fit_iv_weak.raw_model.std_errors['education']:>10.3f}")

                              F-stat    OLS est   2SLS est    2SLS SE
Strong instrument           160998.5      2.444      2.001     0.0022
Weak instrument                 4.41      2.800      2.229      0.335


## Takeaway

With a strong instrument, 2SLS cleanly recovers the true coefficient (2.0) despite naive OLS being badly biased.

With a weak instrument, the first-stage F-statistic drops well below the Stock-Yogo threshold of 10, and 2SLS becomes both **less accurate** (further from 2.0) and **far less precise** (standard error roughly 100x larger). "Use an instrument" is not sufficient on its own — the instrument has to be strong enough for the resulting estimate to actually be trustworthy. Always check the first-stage F-statistic before trusting a 2SLS result.